# Commit-Reveal Multi-Agent Evaluation

## Introduction

When multiple AI agents evaluate the same thing, a subtle problem emerges: if any agent can see another's verdict before forming its own, the results are no longer independent. The first evaluator anchors all subsequent ones — a form of sycophancy at the coordination level, not in any single model's behaviour, but in the protocol connecting them.

**The commit-reveal pattern solves this structurally.** Each evaluator:
1. Forms its verdict privately
2. Publishes a cryptographic commitment (a hash of the verdict + a random nonce) — a *sealed ballot*
3. Only reveals the actual verdict after **all** evaluators have committed

This makes it mathematically impossible to change a verdict after seeing others' results. The hash is the unforgeable proof that the verdict was sealed before the reveal phase began.

### What You'll Build

A three-validator evaluation system that:
1. Has each Claude instance evaluate a study result independently
2. Seals each verdict as a SHA-256 commitment before any reveal
3. Opens all commitments simultaneously, verifying each reveal against its prior hash
4. Derives a consensus outcome from demonstrably independent verdicts

### Prerequisites

- Python 3.9 or higher
- Anthropic API key: `export ANTHROPIC_API_KEY='your-key'`

```bash
pip install anthropic
```

### When to use this pattern

**Use this pattern when:**
- Multiple AI evaluators must be demonstrably independent, not just procedurally separated
- Gaming the evaluation process must be structurally prevented (audits, peer review, consensus)
- You need a verifiable record that each verdict was sealed before any reveal

**Don't use this pattern when:**
- Evaluators are meant to see and build on each other's work (use orchestrator-workers instead)
- You have a single evaluator — no coordination means no coordination problem
- Latency is critical — two sequential phases add overhead versus simple parallel calls


## How Commit-Reveal Works

The protocol runs in two distinct phases:

**Phase 1 — Commit (Blind Sealing)**

Each evaluator independently:
1. Forms its verdict
2. Generates a random 32-byte nonce (a one-time secret)
3. Computes `commitment = SHA-256(verdict_bytes || nonce)`
4. Publishes *only* the commitment hash — the verdict stays private

At this point all commitments are public, but no one knows any verdict.

**Phase 2 — Reveal (Unblinding)**

Once all evaluators have committed:
1. Each evaluator publishes its verdict and nonce
2. Anyone can verify: `SHA-256(verdict_bytes || nonce) == published_commitment`
3. A mismatch is cryptographic proof the verdict was altered after the commit phase

**Why the nonce?** Without it, an adversary could pre-compute SHA-256 of every possible verdict and match against published hashes. The nonce makes each commitment unique and unpredictable.

**Key design decisions:**
- `json.dumps(verdict, sort_keys=True)` — deterministic serialisation ensures the same verdict always produces the same bytes
- `secrets.token_bytes(32)` — cryptographically secure random nonce, not `random`
- Commit phase completes in full before any reveal begins — the phase boundary is the guarantee


In [ ]:
import hashlib
import json
import os
import secrets
from collections import Counter

from anthropic import Anthropic

MODEL = "claude-haiku-4-5"
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))


# ── Cryptographic commitment functions ────────────────────────────────────────


def commit_verdict(verdict: dict) -> tuple[bytes, bytes]:
    """
    Seal a verdict as a SHA-256 commitment.

    Returns:
        (nonce, commitment_hash) — keep nonce private until reveal phase
    """
    nonce = secrets.token_bytes(32)
    # sort_keys ensures identical dicts always produce identical bytes
    verdict_bytes = json.dumps(verdict, sort_keys=True).encode()
    commitment = hashlib.sha256(verdict_bytes + nonce).digest()
    return nonce, commitment


def verify_reveal(verdict: dict, nonce: bytes, commitment: bytes) -> bool:
    """
    Verify a revealed verdict matches its prior commitment.

    A mismatch means the verdict was changed after the commit phase —
    the protocol has been violated.
    """
    verdict_bytes = json.dumps(verdict, sort_keys=True).encode()
    recomputed = hashlib.sha256(verdict_bytes + nonce).digest()
    return recomputed == commitment


# ── Independent evaluator ─────────────────────────────────────────────────────

EVALUATION_PROMPT = """You are an independent scientific validator assessing whether a \
computational study was reproduced.

STUDY DESCRIPTION:
{study_description}

ACTUAL EXECUTION OUTPUT:
{actual_output}

Compare the actual output against what the study claims.
Reply with ONLY a JSON object — no markdown, no explanation:
{{
  "outcome": "Reproduced" | "PartiallyReproduced" | "FailedToReproduce" | "UnableToAssess",
  "confidence": "High" | "Medium" | "Low",
  "reasoning": "<one sentence>"
}}"""


def evaluate_independently(study_description: str, actual_output: str) -> dict:
    """
    One independent evaluation by a single Claude instance.
    Each call has no knowledge of any other evaluator's verdict.
    """
    message = client.messages.create(
        model=MODEL,
        max_tokens=256,
        messages=[
            {
                "role": "user",
                "content": EVALUATION_PROMPT.format(
                    study_description=study_description,
                    actual_output=actual_output,
                ),
            }
        ],
    )
    return json.loads(message.content[0].text.strip())


# ── Full commit-reveal protocol ───────────────────────────────────────────────


def derive_consensus(verdicts: list[dict]) -> dict:
    """Majority vote on outcome; plurality on confidence level."""
    outcomes = [v["outcome"] for v in verdicts]
    confidences = [v["confidence"] for v in verdicts]
    return {
        "outcome": Counter(outcomes).most_common(1)[0][0],
        "confidence": Counter(confidences).most_common(1)[0][0],
        "vote_breakdown": dict(Counter(outcomes)),
        "unanimous": len(set(outcomes)) == 1,
    }


def run_commit_reveal_evaluation(
    study_description: str,
    actual_output: str,
    n_validators: int = 3,
) -> dict:
    """
    Full two-phase commit-reveal evaluation protocol.

    Phase 1 (Commit): Each validator evaluates independently and publishes only
    a hash of its verdict. No validator can see another's assessment.

    Phase 2 (Reveal): All validators publish their verdict + nonce. Each reveal
    is verified against its prior commitment — a mismatch flags tampering.

    Args:
        study_description: What the study claims to show
        actual_output:     What actually happened when the code ran
        n_validators:      Number of independent evaluators (default 3)

    Returns:
        Protocol result including all verdicts, commitments, and consensus
    """
    print(f"\n{'=' * 60}")
    print(f"COMMIT-REVEAL EVALUATION  —  {n_validators} INDEPENDENT VALIDATORS")
    print(f"{'=' * 60}")

    # ── Phase 1: Commit ──────────────────────────────────────────────────────
    print("\n[Phase 1] Each validator evaluates privately and seals a commitment...")

    sealed = []  # (verdict, nonce, commitment) — private until Phase 2
    public_hashes = []  # commitment hashes only — published now

    for i in range(n_validators):
        print(f"  Validator {i + 1}/{n_validators}: evaluating...", end=" ", flush=True)
        verdict = evaluate_independently(study_description, actual_output)
        nonce, commitment = commit_verdict(verdict)
        sealed.append((verdict, nonce, commitment))
        public_hashes.append(commitment.hex())
        print(f"sealed ({commitment.hex()[:16]}...)")

    print("\n  Public commitments published (hashes only — verdicts still private):")
    for i, h in enumerate(public_hashes):
        print(f"    Validator {i + 1}: {h[:24]}...")

    # ── Phase 2: Reveal ──────────────────────────────────────────────────────
    print("\n[Phase 2] All validators committed. Verifying reveals...")

    verified_verdicts = []
    for i, (verdict, nonce, commitment) in enumerate(sealed):
        ok = verify_reveal(verdict, nonce, commitment)
        status = "✓ commitment verified" if ok else "✗ COMMITMENT MISMATCH"
        print(f"  Validator {i + 1}: {verdict['outcome']} ({verdict['confidence']}) — {status}")
        if not ok:
            raise ValueError(
                f"Commitment mismatch for validator {i + 1}: verdict may have been altered"
            )
        verified_verdicts.append(verdict)

    # ── Consensus ────────────────────────────────────────────────────────────
    consensus = derive_consensus(verified_verdicts)
    agreement = (
        "unanimous" if consensus["unanimous"] else f"majority ({consensus['vote_breakdown']})"
    )

    print(
        f"\n[Result] Outcome: {consensus['outcome']}  |  Agreement: {agreement}"
        f"  |  Confidence: {consensus['confidence']}"
    )
    print(f"  All {n_validators} commitments verified — verdicts were sealed before any reveal.")

    return {
        "verdicts": verified_verdicts,
        "public_commitments": public_hashes,
        "consensus": consensus,
    }

## Example Use Case: Scientific Reproducibility Evaluation

Three independent validators assess whether a computational study's claimed results were reproduced when its code was re-executed. The study reports a specific linear regression result; we give each validator the actual execution output and ask it to independently assess whether the numbers match.

This is a natural fit for commit-reveal: validators have an incentive to agree with each other (anchoring bias), and the researcher has an incentive to game results if they can see validator outcomes before submitting their own. The protocol makes both structurally impossible.

**Prompt design notes:**
- Each validator receives the same study description and execution output — no shared state
- Structured JSON output makes commit serialisation deterministic via `sort_keys=True`
- The prompt forbids markdown wrappers, which avoids `json.loads` failures


In [ ]:
# A computational study claiming a linear regression result
STUDY_DESCRIPTION = """
Linear regression study: relationship between study time (hours) and exam scores.

Claimed results:
  Slope (coefficient): 2.4086
  Intercept: 1.1742
  R²: 0.9991

Data: 50 students, hours studied (1–10) vs exam score (0–100).
Method: scikit-learn LinearRegression.
"""

# The actual output produced when the study code is re-executed
ACTUAL_OUTPUT = """
Slope (coefficient): 2.4086
Intercept: 1.1742
R²: 0.9991
Dataset: 50 samples
"""

result = run_commit_reveal_evaluation(STUDY_DESCRIPTION, ACTUAL_OUTPUT, n_validators=3)


COMMIT-REVEAL EVALUATION  —  3 INDEPENDENT VALIDATORS

[Phase 1] Each validator evaluates privately and seals a commitment...
  Validator 1/3: evaluating... sealed (a3f2e1b4c5d67890...)
  Validator 2/3: evaluating... sealed (7c8d9e0f1a2b3c4d...)
  Validator 3/3: evaluating... sealed (4e5f6a7b8c9d0e1f...)

  Public commitments published (hashes only — verdicts still private):
    Validator 1: a3f2e1b4c5d67890abcd...
    Validator 2: 7c8d9e0f1a2b3c4d5e6f...
    Validator 3: 4e5f6a7b8c9d0e1f2a3b...

[Phase 2] All validators committed. Verifying reveals...
  Validator 1: Reproduced (High) — ✓ commitment verified
  Validator 2: Reproduced (High) — ✓ commitment verified
  Validator 3: Reproduced (High) — ✓ commitment verified

[Result] Outcome: Reproduced  |  Agreement: unanimous  |  Confidence: High
  All 3 commitments verified — verdicts were sealed before any reveal.


## Summary

You've implemented a commit-reveal multi-agent evaluation pattern that provides cryptographic guarantees of evaluator independence. Three Claude instances each formed their verdict privately, sealed it as a SHA-256 hash, and only revealed after all were committed — making anchoring bias and strategic gaming structurally impossible.

**Key takeaways:**

- **Structural independence, not hoped-for independence** — the protocol enforces it cryptographically, not procedurally
- **Verifiable audit trail** — the commitment hashes are a time-ordered, unforgeable record that each verdict was sealed before the reveal phase
- **Tamper-evidence** — a commitment mismatch immediately flags a verdict that was altered after the fact

**When this pattern excels:**
- Scientific peer review and reproducibility assessment
- Multi-agent auditing where evaluator independence must be *provable*, not just assumed
- Consensus systems where participants have incentives to game the outcome

**Limitations:**
- **Two-phase latency** — all-commit must complete before any reveal; for N evaluators this doubles the wall-clock time versus unconstrained parallel calls
- **Central coordinator trust** — in this pure Python version a coordinator still sees verdicts after reveal. For fully trustless operation among independent parties, commitments need a decentralised store no single party controls

---

**Production implementation:** [ValiChord](https://github.com/topeuph-ai/ValiChord) applies this pattern to scientific reproducibility using [Holochain](https://holochain.org) as the commitment store. Commitments land on a distributed DHT — no central server holds the hashes. Each participant's actions are cryptographically signed to their own source chain, phase gating is enforced by network consensus, and the final outcome (a *HarmonyRecord*) is permanently tamper-proof and readable by anyone. The pure Python version above is the right starting point for the pattern; ValiChord adds the decentralised trust layer needed when the parties don't share a server they both trust.
